In [4]:
import os
import random
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

# ------------------------------------------------------------
# 1. Sets & basic parameters
# ------------------------------------------------------------
numberCourses = 25
numberRooms = 10
numberSlots = 24
numberDays = 4
slotsPerDay = 6
numberStudents = 400

courses = [f"CS_{100 + i*10}" for i in range(numberCourses)]
rooms = [f"Room_{101 + i}" for i in range(numberRooms)]
timeSlots = [f"Slot_{t+1}" for t in range(numberSlots)]

dayOfSlot = {timeSlots[t]: t // slotsPerDay + 1 for t in range(numberSlots)}

# consecutive same-day slots
adjSlots = [
    (timeSlots[t], timeSlots[t + 1])
    for t in range(numberSlots - 1)
    if (t + 1) % slotsPerDay != 0
]

# ------------------------------------------------------------
# 2. Course enrollments & room capacities
# ------------------------------------------------------------
capacityList = [60, 80, 100, 120, 140, 160, 180, 200, 220, 250]
roomCapacities = {rooms[i]: capacityList[i] for i in range(numberRooms)}

# ------------------------------------------------------------
# 3. Simulate student and course enrollments
# ------------------------------------------------------------
studentID = [f"S_{1000 + i}" for i in range(numberStudents)]
enrollmentMatrix = np.zeros((numberStudents, numberCourses), dtype=int)

for s_idx in range(numberStudents):
    enrolled_course_indices = np.random.choice(
        numberCourses, size=np.random.randint(3, 5), replace=False
    )
    enrollmentMatrix[s_idx, enrolled_course_indices] = 1

studentCourseDataFrame = pd.DataFrame(
    enrollmentMatrix, index=studentID, columns=courses
)
courseEnrollments = {
    c: int(studentCourseDataFrame[c].sum()) for c in courses
}

studentCourses = {
    s: studentCourseDataFrame.columns[
        studentCourseDataFrame.loc[s] == 1
    ].tolist()
    for s in studentID
}

# ------------------------------------------------------------
# 4. Course-to-course conflict parameter
# ------------------------------------------------------------
overlap_dict = {}
for i, c1 in enumerate(courses):
    for j, c2 in enumerate(courses):
        if i < j:
            shared_students = int(
                np.sum(studentCourseDataFrame[c1] & studentCourseDataFrame[c2])
            )
            if shared_students > 0:
                overlap_dict[(c1, c2)] = shared_students

print("--- Data Simulation Complete ---")
print(
    f"Courses: {numberCourses} | Rooms: {numberRooms} | Time Slots:"
    f" {numberSlots} | Students: {numberStudents}"
)


# ============================================================
# 5. Optimization Model
# ============================================================
model = gp.Model("University_Exam_Scheduling")

# valid room-course pairs based on capacity
valid_crt = [
    (c, r, t)
    for c in courses
    for r in rooms
    if courseEnrollments[c] <= roomCapacities[r]
    for t in timeSlots
]

valid_rooms_for_course = {
    c: [r for r in rooms if courseEnrollments[c] <= roomCapacities[r]]
    for c in courses
}

# decision variables
x = model.addVars(valid_crt, vtype=GRB.BINARY, name="x")

adj_slot_starts = [t1 for t1, _ in adjSlots]
y = model.addVars(studentID, adj_slot_starts, vtype=GRB.BINARY, name="y")

# objective: minimize back-to-back exams
model.setObjective(
    gp.quicksum(y[s, t1] for s in studentID for t1 in adj_slot_starts),
    GRB.MINIMIZE,
)

# each course gets 1 room and 1 slot
model.addConstrs(
    (
        gp.quicksum(
            x[c, r, t] for r in valid_rooms_for_course[c] for t in timeSlots
        )
        == 1
        for c in courses
    ),
    name="OneAssignmentPerCourse",
)

# max 1 exam per room per slot
model.addConstrs(
    (
        gp.quicksum(
            x[c, r, t]
            for c in courses
            if courseEnrollments[c] <= roomCapacities[r]
        )
        <= 1
        for r in rooms
        for t in timeSlots
    ),
    name="RoomMutex",
)

# no overlapping exams for courses with shared students
model.addConstrs(
    (
        gp.quicksum(x[c1, r, t] for r in valid_rooms_for_course[c1])
        + gp.quicksum(x[c2, r, t] for r in valid_rooms_for_course[c2])
        <= 1
        for (c1, c2) in overlap_dict
        for t in timeSlots
    ),
    name="PairwiseConflict",
)

# track back-to-back exams on the same day
model.addConstrs(
    (
        gp.quicksum(
            x[c, r, t1] + x[c, r, t2]
            for c in studentCourses[s]
            for r in valid_rooms_for_course[c]
        )
        - 1
        <= y[s, t1]
        for s in studentID
        for (t1, t2) in adjSlots
        if len(studentCourses[s]) >= 2
    ),
    name="BackToBackTracking",
)

# ------------------------------------------------------------
# 6. Solve Model
# ------------------------------------------------------------
print("Optimizing model with Gurobi...")

# Give Gurobi a 15-second time limit so it doesn't freeze your notebook
model.Params.TimeLimit = 15 
# Allow a 5% optimality gap so it stops as soon as it finds a great solution
model.Params.MIPGap = 0.05 

model.optimize()

# ------------------------------------------------------------
# 7. Export Results & Outputs
# ------------------------------------------------------------
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

studentCourseDataFrame.to_csv("data/student_enrollments.csv")
conflictDataFrame = pd.DataFrame(
    [(c1, c2, count) for (c1, c2), count in overlap_dict.items()],
    columns=["Course 1", "Course 2", "Shared Students"],
)
conflictDataFrame.to_csv("data/course_conflicts.csv", index=False)

# Check if Gurobi found at least one valid schedule (even if hit time limit)
if model.SolCount > 0:
    print(f"\nSolution found! Status: {model.status}")
    print(f"Total Back-to-Back Student Exam Instances: {int(round(model.ObjVal))}")
    print(f"Solve Time: {model.Runtime:.4f} seconds")

    schedule_records = []
    for c, r, t in valid_crt:
        if x[c, r, t].X > 0.5:
            schedule_records.append({
                "Course": c,
                "Enrollment": courseEnrollments[c],
                "Assigned_Room": r,
                "Room_Capacity": roomCapacities[r],
                "Time_Slot": t,
                "Day": dayOfSlot[t],
            })

    df_schedule = pd.DataFrame(schedule_records)
    df_schedule.to_csv("outputs/optimal_exam_schedule.csv", index=False)
    print(
        "Exam schedule exported successfully to 'outputs/optimal_exam_schedule.csv'!"
    )
    print(df_schedule.head(10))
else:
    print(f"\nNo solution found. Optimization status {model.status}.")

--- Data Simulation Complete ---
Courses: 25 | Rooms: 10 | Time Slots: 24 | Students: 400
Optimizing model with Gurobi...
Set parameter TimeLimit to value 15
Set parameter MIPGap to value 0.05
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  15
MIPGap  0.05

Optimize a model with 15441 rows, 13904 columns and 704864 nonzeros (Min)
Model fingerprint: 0xbc59c283
Model has 8000 linear objective coefficients
Variable types: 0 continuous, 13904 integer (13904 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 135.0000000
Presolve removed 7348 rows and 5092 columns
Presolve time: 2.39s
Presolved: 8093 rows, 8812 columns, 105756 nonzeros
Variable types: 0 continuous, 8812 inte